<a href="https://colab.research.google.com/github/ikabrain/UCS772-CV-NLP-Lab/blob/main/NLP_assign6/NLP_assign6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 6 - Text prediction using advanced method
---

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

SEED = 42

## Preprocessing Text
---

### Initialising Dataset

In [2]:
import kagglehub
path = kagglehub.dataset_download("ashishpandey2062/next-word-predictor-text-generator-dataset")

Using Colab cache for faster access to the 'next-word-predictor-text-generator-dataset' dataset.


In [3]:
!ls {path}

next_word_predictor.txt


In [4]:
with open(path + "/next_word_predictor.txt", 'r', encoding='utf-8') as file:
    data = file.read()

print(data[:500])

The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.

As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air.


In [5]:
lines = [l.strip() for l in data.split("\n") if l.strip()]
print(lines[:3])

['The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.', 'As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.', 'In the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.']


### Exploring Corpus

In [6]:
import re
from collections import Counter

In [7]:
# Overall size
print("chars:", len(data))
print("lines:", len(lines), "| empty:", sum(1 for l in lines if not l.strip()))
print("words:", len(data.split()))

chars: 167445
lines: 1547 | empty: 0
words: 27644


In [8]:
# Most frequent words
words = re.findall(r"[a-z']+", data.lower())
print(Counter(words).most_common(50))

[('the', 1418), ('and', 859), ('a', 718), ('of', 689), ('to', 596), ('i', 431), ('you', 380), ('in', 307), ('is', 272), ('monica', 267), ('it', 248), ('with', 246), ('ross', 245), ('that', 238), ('rachel', 202), ('for', 187), ('chandler', 180), ('this', 166), ('on', 159), ('joey', 151), ('was', 144), ('oh', 136), ('phoebe', 135), ('are', 129), ('all', 123), ('as', 120), ('what', 119), ('be', 111), ('like', 111), ('no', 108), ("it's", 106), ("i'm", 103), ('her', 100), ('they', 98), ('just', 98), ('from', 95), ('okay', 94), ('not', 93), ('so', 89), ('my', 89), ('have', 87), ('me', 86), ('where', 83), ('she', 81), ('know', 80), ('we', 79), ('out', 77), ('well', 77), ('their', 77), ('can', 77)]


In [9]:
# Shortest non-empty lines
nonempty = [l.strip() for l in lines if l.strip()] # Redundant as no empty line is read in this case!
for l in sorted(set(nonempty), key=len)[:40]:
    print(repr(l))

'End'
'Usage'
'(Pause)'
'Encoding'
'Ross:Hi.'
'Kid: No.'
'All: Mmm.'
'Alan: Hi.'
'All: Yes!'
'Ross: Hi.'
'Joey: So?'
'All: Eww!'
'Ross: Mm.'
'Joey: Ohh.'
'Susan: Hi.'
'All: Nooo!'
'Ross: Why?'
'Alan: Wow.'
'Carol: So.'
'Paula: No.'
'All: Yeah!'
'Joey: Hey!'
'(Silence.)'
'All: What?'
'Joey: What?'
'Susan: She.'
'Ross: What?'
'Ross: Easy.'
'Monica: No.'
'[SCENE END]'
'Ross: Ohhh.'
'Paul: Yeah?'
'Phoebe: No.'
'Characters:'
'Ross: Yeah.'
'Joey: Alan.'
'Joey: Okay!'
'All: Eeaagh!'
'Phoebe: Huh!'
'Robbie: Me?!'


We see that there's a script of the TV show "Friends" here with speaker labels, that can give us some trouble. Let's handle that.

In [10]:
# Catch speaker labels
prefixes = Counter(m.group(1) for l in nonempty
                   if (m := re.match(r"^([A-Z][\w .,'-]{0,30}):", l)))
print(prefixes.most_common(30))

[('Monica', 200), ('Ross', 183), ('Rachel', 155), ('Chandler', 123), ('Phoebe', 102), ('Joey', 99), ('All', 35), ('Headline', 24), ('Carol', 21), ('Paul', 17), ('Barry', 16), ('Lizzie', 11), ('Sarah', 10), ('Mrs. Geller', 10), ('Mark', 9), ('Susan', 9), ('Pizza Guy', 7), ('Paula', 6), ('Alan', 6), ('Receptionist', 6), ('Frannie', 5), ('Mr. Geller', 5), ('Robbie', 5), ('Joanne', 5), ('Written by', 4), ('Marsha', 4), ('Kiki', 4), ('Main article', 3), ('Leslie', 3), ('Kid', 3)]


In [11]:
# Bracketed content
brackets = re.findall(r"\([^)]*\)|\[[^\]]*\]", data)
print("Parenthesized content: ", len(brackets))
print(brackets[:5], "..." if len(brackets) > 5 else "")
print()

# Unbalanced parentheses within a line
bad = [l for l in nonempty if l.count("(") != l.count(")")]
print("No. of lines with unbalanced parethesis: ", len(bad))
print(bad[:5], "..." if len(bad) > 5 else "")

Parenthesized content:  449
['(AI)', '(BCI)', '(10,000 words continued)', '(IoT)', '[SCENE START]'] ...

No. of lines with unbalanced parethesis:  1
['Rachel: Oh my God! (Rachel, Leslie, Kiki, and Joanne all scream and hug each other.'] 


In [12]:
# Non-ASCII characters
non_ascii = Counter(c for c in data if ord(c) > 127)
print(non_ascii.most_common())

[('—', 11), ('í', 5), ('á', 2), ('…', 2), ('ö', 1), ('¬', 1), ('–', 1)]


In [13]:
# Punctuation characters
punct = Counter(c for c in data if not c.isalnum() and not c.isspace())
print(punct.most_common())

[('.', 2513), (',', 2233), (':', 1198), ("'", 1007), ('?', 431), ('-', 384), ('(', 379), (')', 378), ('!', 377), ('"', 88), ('[', 73), (']', 73), (';', 12), ('—', 11), ('/', 9), ('+', 4), ('&', 3), ('…', 2), ('%', 1), ('$', 1), ('¬', 1), ('`', 1), ('=', 1), ('–', 1)]


In [14]:
# Duplicate lines
dupes = [l for l, c in Counter(nonempty).items() if c > 1]
print("Exact duplicate lines:", len(dupes))
print(dupes[:5], "..." if len(dupes) > 5 else "")

# Near-duplicate lines with same starts
starts = Counter(l[:40].lower() for l in nonempty if len(l) > 40)
print([s for s, c in starts.items() if c > 1])

Exact duplicate lines: 13
['Note: This is a sample of the 10,000-word dataset. The full dataset will be generated upon confirmation.', 'Characters:', '[SCENE START]', 'Written by: Marta Kauffman & David Crane', 'Commercial Break'] ...
['explorers ventured deep into the amazon,', 'note: this is a sample of the 10,000-wor', "[scene: monica and rachel's, everyone is", '[scene: madison square garden, the guys ', "[scene: monica and rachel's, the girls a"]


In [15]:
# Looking for meta-text from LLM yapping
meta = re.compile(r"rnn model|dataset|certainly|please let me know|note:|sample", re.I)
hits = [l for l in nonempty if meta.search(l)]
print("Found meta-text:", len(hits))
print()
for l in hits:
    print("-", l[:100])

Found meta-text: 17

- Your RNN model can use this text to predict the next word in a sequence, offering an exciting opport
- The Amazon was a place of awe and wonder, teeming with life and secrets waiting to be uncovered. You
- The Amazon was a realm of magic and mystery, a treasure trove of life and enigma waiting to be unrav
- The Faroe Islands were a place of rugged beauty, where human resilience and the forces of nature int
- Machu Picchu was a sanctuary of history, a place where the past and present intertwined. Your RNN mo
- Iceland was a realm of elemental grandeur, a place where the forces of nature shaped every aspect of
- The Sahara was a realm of solitude and wonder, where humans and nature coexisted in a delicate balan
- Bhutan was a realm of spiritual harmony and environmental stewardship, where ancient traditions and 
- Certainly, here's another unique and extended text passage for your next word predictor project:
- Madagascar was a realm of evolution's marvels and cult

### Text Cleaning

In [16]:
DROP_STARTS = ("certainly", "note:", "please let me know", "this is just", "this sample", "tech news dataset", "written by", "remember, the key")

In [17]:
import re, unicodedata

def clean_line(line):
    line = re.sub(r"Your RNN model[^.]*\.", " ", line)       # meta sentence at the end of paragraphs
    line = re.sub(r"\(.*?\)|\[.*?\]", " ", line)             # stage directions, [Scene: ...], [1], etc etc
    line = re.sub(r"\(.*$", " ", line)                       # unclosed "(" runs to end of line
    line = line.replace(")", " ")                            # stray ")"
    line = re.sub(r"^[A-Z][\w .,'-]{0,30}:\s*", "", line)    # "Monica: ", "Headline: ", "Setting: ", other speaker calls
    line = line.replace("\u2019", "'").replace("\u2018", "'")
    line = re.sub(r"\u2014|\u2013|\u2026|--", " ", line)     # handling dashes and the ellipsis character
    line = unicodedata.normalize("NFKD", line).encode("ascii", "ignore").decode()  # García -> Garcia
    return re.sub(r"\s+", " ", line).strip()

In [18]:
def clean_text(data):
    out = []
    for line in data.split("\n"):
        if line.strip().lower().startswith(DROP_STARTS):
            continue
        line = clean_line(line)
        if len(line.split()) >= 3:      # also drops "End", "Commercial Break" and empty lines
            out.append(line)
    return out

In [19]:
# Final cleaned text
cleaned = clean_text(data)
print("No. of cleaned lines: ", len(cleaned))
print("- " + "\n- ".join(cleaned[:10]))

No. of cleaned lines:  1164
- The sun was shining brightly in the clear blue sky, and a gentle breeze rustled the leaves of the tall trees. People were out enjoying the beautiful weather, some sitting in the park, others taking a leisurely stroll along the riverbank. Children were playing games, and laughter filled the air.
- As the day turned into evening, the temperature started to drop, and the sky transformed into a canvas of vibrant colors. Families gathered for picnics, and the smell of barbecues wafted through the air. It was a perfect day for a picnic by the lake.
- In the distance, you could hear the sound of live music coming from a local band, and people began to gather around the stage to enjoy the performance. The atmosphere was electric, and the music had everyone swaying to the beat.
- As the stars began to twinkle in the night sky, the crowd grew even larger, and the festivities continued well into the night. It was a day filled with joy, laughter, and memories that wou

In [20]:
lines = list(dict.fromkeys(cleaned)) # Drop exact duplicate lines, BUT keep order!

### Train-Val-Test Sentence Split

In [21]:
rng = np.random.default_rng(seed=SEED)
rng.shuffle(lines)

In [22]:
n = len(lines)
i1, i2 = int(0.8 * n), int(0.9 * n) # Using 80-10-10 split
train_lines, val_lines, test_lines = lines[:i1], lines[i1:i2], lines[i2:]

for name, part in {"training": train_lines, "validation": val_lines, "testing": test_lines}.items():
    print(name.capitalize()+":", len(part), "lines,", sum(len(l.split()) for l in part), "words")

Training: 928 lines, 18558 words
Validation: 116 lines, 2319 words
Testing: 117 lines, 1843 words


In [23]:
# Sentence-splitting
def to_sentences(lines):
    out = []
    for line in lines:
        line = re.sub(r"\b(Mr|Mrs|Ms|Dr|St)\.", r"\1", line)   # don't split on titles
        for s in re.split(r"(?<=[.!?])\s+", line):
            if len(s.split()) >= 2:
                out.append(s)
    return out

train_sents = to_sentences(train_lines)
val_sents   = to_sentences(val_lines)
test_sents  = to_sentences(test_lines)
print("Training:", len(train_sents), "sentences")
print("Validation:", len(val_sents), "sentences")
print("Testing:", len(test_sents), "sentences")

Training: 1698 sentences
Validation: 208 sentences
Testing: 180 sentences


In [24]:
# Checking sentence lengths for max sequence length
lens = np.array([len(s.split()) for s in train_sents])
print("Training has", len(lens), "sentences.")
print("Length percentiles (50/90/95/99/max):", np.percentile(lens, [50, 90, 95, 99]), lens.max())

Training has 1698 sentences.
Length percentiles (50/90/95/99/max): [ 8.   22.   26.   34.03] 53


### Tokenization Layer

In [25]:
# Size of vocabulary across training sents for max tokens
counts = Counter(w for s in train_sents for w in re.findall(r"[a-z0-9']+", s.lower()))
print("Training has", len(counts), "unique words.")
print("Words seen at least twice:", sum(1 for c in counts.values() if c >= 2))

Training has 4022 unique words.
Words seen at least twice: 1794


In [26]:
MAX_WORDS = 2000
SEQ_LEN = 30
PAD, UNK, START, INDEX_FROM = 0, 1, 2, 3

In [27]:
def standardize(x):
    """
    Standardizes given string of text and appends a starting token '[BOS]'
    """
    x = tf.strings.lower(x)
    x = tf.strings.regex_replace(x, r"[^a-z0-9\s']", " ")
    x = tf.strings.regex_replace(x, r"(^|\s)'+|'+(\s|$)", " ")
    x = tf.strings.strip(tf.strings.regex_replace(x, r"\s+", " "))
    return tf.strings.join(["[BOS]", x], separator=" ")

In [28]:
vectorizer = layers.TextVectorization(
    max_tokens=MAX_WORDS+3,
    standardize=standardize,
    output_mode="int",
    output_sequence_length=SEQ_LEN+1, # xxbos + 30 words; makes sure to post-pad and post-truncate too!
)

vectorizer.adapt(train_sents)

vocab = vectorizer.get_vocabulary()
vocab[:10]

['',
 '[UNK]',
 np.str_('[BOS]'),
 np.str_('the'),
 np.str_('and'),
 np.str_('a'),
 np.str_('of'),
 np.str_('to'),
 np.str_('i'),
 np.str_('you')]

### Generating Tokenized Input Sequences

In [29]:
def build_ngram_data(sents):
    ids = vectorizer(tf.constant(sents)).numpy() # (N, SEQ_LEN+1), padded with 0 AT THE END

    input_sequences = []
    for row in ids:
        tokenized_sentence = row[row != 0].tolist() # Strip padding
        for i in range(1, len(tokenized_sentence)):
            input_sequences.append(tokenized_sentence[:i + 1])

    padded = keras.utils.pad_sequences(input_sequences, maxlen=SEQ_LEN+1, padding="pre") # NOTE: Same maxlen for every split
    X, y = padded[:, :-1], padded[:, -1]
    keep = (y != UNK) # [UNK] = id 1, never a target worth learning!
    return X[keep], y[keep]

In [30]:
X_train, y_train = build_ngram_data(train_sents)
X_val,   y_val   = build_ngram_data(val_sents)
X_test,  y_test  = build_ngram_data(test_sents)

In [31]:
# X_train sentence ngram -> y_train next expected word
for i in range(5):
    print([vocab[t] for t in X_train[i] if t != 0], "->", vocab[y_train[i]])

[np.str_('[BOS]')] -> well
[np.str_('[BOS]'), np.str_('well')] -> i'm
[np.str_('[BOS]'), np.str_('well'), np.str_("i'm")] -> going
[np.str_('[BOS]'), np.str_('well'), np.str_("i'm"), np.str_('going')] -> out
[np.str_('[BOS]'), np.str_('well'), np.str_("i'm"), np.str_('going'), np.str_('out')] -> with


## Q1: Apply deep learning models i.e. LSTM for next word prediction for sentiment analysis data
---

### Building the Model

In [32]:
VOCAB_SIZE = len(vocab) # 2000 words + '', [UNK], [BOS]
keras.utils.set_random_seed(SEED)

In [33]:
model = Sequential(
    [
        layers.Input(shape=(SEQ_LEN,)),
        layers.Embedding(VOCAB_SIZE, 100),  # NOTE: We have no mask_zero, as pre-padding already puts real words last!
        layers.LSTM(128),
        layers.Dropout(0.3),
        layers.Dense(VOCAB_SIZE, activation="softmax"),
    ]
)

In [34]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy", # !WARN: y is word ids, not one-hot!!!
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5")],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 100)        │       200,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       117,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2003)           │       258,387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 575,935 (2.20 MB)

 Trainable params: 575,935 (2.20 MB)

 Non-trainable params: 0 (0.00 B)

### Callbacks

In [35]:
callbacks = [
    ModelCheckpoint("best_model.keras", monitor="val_loss", save_best_only=True, verbose=0),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=2, min_lr=1e-6, verbose=1), # patience < EarlyStopping's!
]

### Training

In [36]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val), # NOT test!
    epochs=50,                      # upper bound, EarlyStopping decides
    batch_size=64,
    callbacks=callbacks,
    verbose=2,                      # one line per epoch
)

Epoch 1/50
256/256 - 8s - 30ms/step - accuracy: 0.0573 - loss: 6.4468 - top5: 0.1741 - val_accuracy: 0.0739 - val_loss: 5.8842 - val_top5: 0.2300 - learning_rate: 0.0010
Epoch 2/50
256/256 - 2s - 8ms/step - accuracy: 0.0724 - loss: 6.0817 - top5: 0.1948 - val_accuracy: 0.0922 - val_loss: 5.7594 - val_top5: 0.2344 - learning_rate: 0.0010
Epoch 3/50
256/256 - 2s - 8ms/step - accuracy: 0.0820 - loss: 5.8961 - top5: 0.2089 - val_accuracy: 0.1039 - val_loss: 5.6527 - val_top5: 0.2489 - learning_rate: 0.0010
Epoch 4/50
256/256 - 2s - 7ms/step - accuracy: 0.0954 - loss: 5.7151 - top5: 0.2214 - val_accuracy: 0.1161 - val_loss: 5.5363 - val_top5: 0.2544 - learning_rate: 0.0010
Epoch 5/50
256/256 - 2s - 8ms/step - accuracy: 0.1095 - loss: 5.5369 - top5: 0.2399 - val_accuracy: 0.1228 - val_loss: 5.4418 - val_top5: 0.2722 - learning_rate: 0.0010
Epoch 6/50
256/256 - 2s - 8ms/step - accuracy: 0.1206 - loss: 5.3733 - top5: 0.2568 - val_accuracy: 0.1267 - val_loss: 5.3791 - val_top5: 0.2694 - learnin

### Generating Text by Predicting Next Words

In [37]:
def generate_text(text, n_words=15, k=5, temperature=0.8):
    """
    Takes in a prompt string, no. of words to generate, top-k cutoff and temperature
    Returns a generator yielding one sampled word at a time
    """
    ids = [i for i in vectorizer([text]).numpy()[0] if i != 0] # [BOS] + prompt, padding stripped
    for _ in range(n_words):
        x = keras.utils.pad_sequences([ids], maxlen=SEQ_LEN)
        probs = model(x, training=False)[0].numpy().astype("float64") # float64, else rng.choice may reject the sum
        top = np.argsort(probs)[-k:] # ids of the k most likely words
        p = probs[top] ** (1 / temperature) # <1 sharpens, >1 flattens
        next_id = int(rng.choice(top, p=p / p.sum())) # renormalise over the top k
        ids.append(next_id)
        yield str(vocab[next_id])

In [45]:
for idx, prompt in enumerate(["the sun was", "i don't know", "the castle had"], start=1):
    print(f"PROMPT {idx}:", prompt, end=" ")
    for word in generate_text(prompt, n_words=20, k=3, temperature = 0.95):
        print(word, end=" ", flush=True)
        time.sleep(0.1)  # typing effect ;)
    print()

PROMPT 1: the sun was a realm of the air and the vibrant energy of ancient life and the air of the sun of the 
PROMPT 2: i don't know you do you guys do you have it to the little thing i mean you have a date with a 
PROMPT 3: the castle had a realm of the vibrant peaks with the sun where the sun and the sun of the sun where the 


## Q2: Comment based on the various evaluation metrics for your results.
---

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay